In [ ]:
import pandas as pd
df = pd.read_csv('Bank_Marketing.csv', sep=';')

# Quick look
print(df.shape)
print(df['y'].value_counts(normalize=True))
df.head()

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Auto-detect correct column names
balance_col = 'balance' if 'balance' in df.columns else 'cons.price.idx'  # fallback (rare)
print("Using balance column:", 'balance' if 'balance' in df.columns else "NOT FOUND – check your file!")

# If you really don't have 'balance', just comment out that line below
score = 0

# Job title
score += df['job'].isin(['admin.','management','entrepreneur','self-employed']).astype(int) * 25

# Age sweet spot
score += df['age'].between(30, 55).astype(int) * 15

# High balance (safe version – skips if column missing)
if 'balance' in df.columns:
    score += (df['balance'] > 5000).astype(int) * 30
else:
    print("Warning: 'balance' column not found – skipping that rule")

# Previous success
score += (df['poutcome'] == 'success').astype(int) * 40

# Housing loan penalty
score -= (df['housing'] == 'yes').astype(int) * 10

# Final
df['lead_score'] = score.clip(0, 150)
df['lead_tier'] = pd.cut(df['lead_score'], bins=[-float('inf'),50,80,float('inf')], labels=['Cold','Warm','Hot'])

print(df['lead_tier'].value_counts().sort_index())
print("\nTop 10 Hot leads:")
print(df.nlargest(10,'lead_score')[['age','job','poutcome','lead_score','lead_tier']])

# Row-level scored exports are intentionally not written in the public notebook.
print("\nSaved local scoring output")




ax = df['lead_tier'].value_counts().sort_index().plot(
    kind='barh',
    
    figsize=(9,3)
)

# Get counts and percentages
counts = df['lead_tier'].value_counts().sort_index()
total = counts.sum()
percentages = counts / total * 100

# Annotate each bar with count and percentage
for i, (count, pct) in enumerate(zip(counts, percentages)):
    ax.text(count + 0.5, i, f'{count} ({pct:.1f}%)', va='center')

plt.title('Project #1 – Bank Marketing Lead Tiers (Rule-Based)')
plt.xlabel('Number of Clients')
plt.tight_layout()
plt.savefig('../images/06_Bar_Chart.png', dpi=160, bbox_inches='tight')
plt.tight_layout()
plt.savefig('../images/06_Bar_Chart.png', dpi=160, bbox_inches='tight')
plt.show()